- focus on filter_item function and saves the rejected items within the loop


In [ ]:
import os
import re
import requests
import pandas as pd
import logging
from dotenv import load_dotenv
from datetime import datetime, timedelta
from time import sleep, time as now
import backoff

# === CONFIG ===
output_dir = r"C:\\Android Mobile App\\Step1_URL_Search\\Filter_Item_Rejections_Check"
os.makedirs(output_dir, exist_ok=True)

# === LOGGING ===
ts_log = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = os.path.join(output_dir, f"log_{ts_log}.log")
logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(asctime)s - %(message)s",
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# === AUTH (with Token Rotation) ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 6)]
tokens = [t for t in tokens if t]

if not tokens:
    raise ValueError("❌ No GitHub tokens found in All_Tokens.env")

token_index = 0

def get_headers():
    return {
        "Authorization": f"token {tokens[token_index]}",
        "Accept": "application/vnd.github.mercy-preview+json",
        "User-Agent": "android-repo-crawler/1.0"
    }

def rotate_token():
    global token_index
    token_index = (token_index + 1) % len(tokens)
    logger.warning(f"🔁 Rotated to token #{token_index + 1}")

@backoff.on_exception(backoff.expo,
                      (requests.exceptions.RequestException, requests.exceptions.Timeout),
                      max_tries=5,
                      jitter=None)
def request_with_retry(url, params=None, timeout=10):
    global token_index
    for _ in range(len(tokens)):
        try:
            response = requests.get(url, headers=get_headers(), params=params, timeout=timeout)
            if response.status_code == 200:
                return response
            elif response.status_code == 403:
                logger.warning(f"🔁 Token #{token_index+1} rate-limited, rotating...")
                rotate_token()
                sleep(2)
            else:
                logger.error(f"❌ Unexpected status: {response.status_code} — {response.text}")
                return response
        except Exception as e:
            logger.warning(f"⚠️ Error during request: {e}")
            rotate_token()
            sleep(2)
    return None

# === DATE RANGE ===
start_date = datetime.strptime("2008-01-01", "%Y-%m-%d")
end_date = datetime.strptime("2024-12-31", "%Y-%m-%d")

# === WINDOW SETTINGS ===
initial_window_hours = 15 * 24
min_window_hours = 1
max_window_hours = 90 * 24
MAX_RESULTS_PER_QUERY = 1000
TARGET_FILL_RATIO = 0.25

# === BASE QUERIES ===
base_queries = [
    "stars:>50 language:Kotlin fork:false archived:false",
    "stars:>50 language:Java fork:false archived:false",
    "stars:>50 language:Dart fork:false archived:false",
    "stars:>50 topic:android fork:false archived:false",
    "stars:>50 android in:name,description,readme fork:false archived:false",
]

# === Filter ===
rejected_items = []
# === Filter ===
rejected_items = []

def filter_item(item, expected_fork, expected_archived, expected_languages):
    min_stars = 50
    passed = True
    rejection_reasons = []
    notes = []
    verified_primary_language = ""

    language = (item.get('language') or '').lower()
    full_name = item.get("full_name", "")

    logger.info(f"🔍 Evaluating {full_name} (lang: {language}, stars: {item.get('stargazers_count')}, fork: {item.get('fork')}, archived: {item.get('archived')})")

    if item.get('stargazers_count', 0) <= min_stars:
        passed = False
        rejection_reasons.append("Below minimum stars")
    if item.get('fork', False) != expected_fork:
        passed = False
        rejection_reasons.append("Fork status mismatch")
    if item.get('archived', False) != expected_archived:
        passed = False
        rejection_reasons.append("Archived status mismatch")

    try:
        owner, repo = full_name.split("/")
        lang_url = f"https://api.github.com/repos/{owner}/{repo}/languages"
        lang_resp = request_with_retry(lang_url)
        if lang_resp is None:
            passed = False
            rejection_reasons.append("Language API fetch failed")
        elif lang_resp.status_code == 200:
            langs = [k.lower() for k in lang_resp.json().keys()]
            matching = [lang for lang in expected_languages if lang in langs]
            if matching:
                verified_primary_language = matching[0]
                if language not in matching:
                    item["language"] = verified_primary_language
                    notes.append(f"Language corrected to '{verified_primary_language}' via /languages")
                    logger.info(f"   📜 Language updated from '{language}' → '{verified_primary_language}'")
            else:
                passed = False
                rejection_reasons.append("Language mismatch (via /languages endpoint)")
        else:
            passed = False
            rejection_reasons.append("Language API fetch failed")
    except Exception as e:
        passed = False
        rejection_reasons.append(f"Language check error: {str(e)}")

    if passed:
        logger.info(f"   ✅ Passed: {full_name}")
    else:
        logger.warning(f"   ❌ Rejected: {full_name} → {', '.join(rejection_reasons)}")

    item["verified_primary_language"] = verified_primary_language

    if not passed:
        rejected_items.append({
            "full_name": full_name,
            "language": item.get("language"),
            "verified_primary_language": verified_primary_language,
            "description": item.get("description"),
            "topics": item.get("topics"),
            "stars": item.get("stargazers_count"),
            "fork": item.get("fork"),
            "archived": item.get("archived"),
            "base_qualifier": item.get("base_qualifier", "N/A"),
            "html_url": item.get("html_url"),
            "rejection_reasons": "; ".join(rejection_reasons),
            "notes": "; ".join(notes)
        })

    return passed

def check_rate_limit():
    r = request_with_retry("https://api.github.com/rate_limit")
    if r and r.status_code == 200:
        data = r.json()
        remaining = data['resources']['search']['remaining']
        reset_epoch = data['resources']['search']['reset']
        reset_in = max(0, reset_epoch - now())
        logger.info(f"🔎 Remaining: {remaining} | Resets in {reset_in/60:.1f} min")
        if remaining < 5:
            logger.warning(f"⏳ Rate limit low. Sleeping {reset_in/60:.1f} min")
            rotate_token()
            sleep(reset_in + 5)

def check_count(query):
    r = request_with_retry("https://api.github.com/search/repositories", params={"q": query, "per_page": 1})
    if r and r.status_code == 200:
        return r.json().get("total_count", 0)
    return -1

def fetch_items(query):
    items = []
    for page in range(1, 11):
        check_rate_limit()
        r = request_with_retry("https://api.github.com/search/repositories", params={"q": query, "per_page": 100, "page": page})
        if r and r.status_code == 200:
            page_items = r.json().get("items", [])
            if not page_items:
                break
            items.extend(page_items)
            sleep(1)
        else:
            break
    return items

# === MAIN EXECUTION ===
final_results = []

for base_query_prefix in base_queries:
    current_start = start_date
    window_hours = initial_window_hours

    while current_start < end_date:
        current_end = min(current_start + timedelta(hours=window_hours), end_date)
        date_range = f"created:{current_start.isoformat()}..{current_end.isoformat()}"
        base_query = f"{base_query_prefix} {date_range}"

        total_count = check_count(base_query)
        logger.info(f"⏳ {base_query} → {total_count} repos")

        if total_count >= MAX_RESULTS_PER_QUERY and window_hours > min_window_hours:
            window_hours = max(window_hours // 2, min_window_hours)
            continue
        elif total_count < MAX_RESULTS_PER_QUERY * TARGET_FILL_RATIO and window_hours * 2 <= max_window_hours:
            window_hours = min(window_hours * 2, max_window_hours)

        items = fetch_items(base_query)

        expected_fork = 'fork:true' in base_query_prefix
        expected_archived = 'archived:true' in base_query_prefix
        expected_language = []

        if 'language:Kotlin' in base_query_prefix:
            expected_language = ['kotlin', 'java', 'dart']
        elif 'language:Java' in base_query_prefix:
            expected_language = ['kotlin', 'java', 'dart']
        elif 'language:Dart' in base_query_prefix:
            expected_language = ['kotlin', 'java', 'dart']

        count_passed = 0
        for item in items:
            item["base_qualifier"] = base_query_prefix
            if filter_item(item, expected_fork, expected_archived, expected_language):
                final_results.append({
                    "name": item.get("name"),
                    "full_name": item.get("full_name"),
                    "language": item.get("language"),
                    "verified_primary_language": item.get("verified_primary_language", ""),
                    "stargazers_count": item.get("stargazers_count"),
                    "forks": item.get("forks"),
                    "topics": item.get("topics"),
                    "fork": item.get("fork"),
                    "archived": item.get("archived"),
                    "owner.login": item.get("owner", {}).get("login"),
                    "html_url": item.get("html_url"),
                    "clone_url": item.get("clone_url"),
                    "visibility": item.get("visibility"),
                    "size": item.get("size"),
                    "open_issues_count": item.get("open_issues_count"),
                    "base_qualifier": base_query_prefix,
                    "search_qualifier": base_query
                })
                count_passed += 1

        logger.info(f"✅ Window done: {count_passed} passed")

        if rejected_items:
            ts = datetime.now().strftime("%Y%m%d_%H%M%S")
            safe_prefix = re.sub(r'[<>:"/\\|?*]', '_', base_query_prefix)
            safe_prefix = safe_prefix.replace(" ", "_").replace(",", "")
            reject_path = os.path.join(output_dir, f"rejected_{safe_prefix}_{ts}.csv")
            pd.DataFrame(rejected_items).to_csv(reject_path, index=False)
            logger.info(f"🚫 Rejected repos saved: {reject_path}")
            rejected_items.clear()

        current_start = current_end + timedelta(seconds=1)

# === SAVE RESULTS ===
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
df = pd.DataFrame(final_results)
df.to_excel(os.path.join(output_dir, f"search_results_{ts}.xlsx"), index=False)
logger.info("✅ Accepted repos saved.")
print(f"✅ Total passed repos: {len(df)}")


[INFO] ⏳ stars:>50 language:Kotlin fork:false archived:false created:2008-01-01T00:00:00..2008-01-16T00:00:00 → 0 repos
[INFO] 🔎 Remaining: 29 | Resets in 1.0 min
[INFO] ✅ Window done: 0 passed
[INFO] ⏳ stars:>50 language:Kotlin fork:false archived:false created:2008-01-16T00:00:01..2008-02-15T00:00:01 → 0 repos
[INFO] 🔎 Remaining: 27 | Resets in 1.0 min
[INFO] ✅ Window done: 0 passed
[INFO] ⏳ stars:>50 language:Kotlin fork:false archived:false created:2008-02-15T00:00:02..2008-04-15T00:00:02 → 0 repos
[INFO] 🔎 Remaining: 25 | Resets in 1.0 min
[INFO] ✅ Window done: 0 passed
[INFO] ⏳ stars:>50 language:Kotlin fork:false archived:false created:2008-04-15T00:00:03..2008-06-14T00:00:03 → 0 repos
[INFO] 🔎 Remaining: 23 | Resets in 1.0 min
[INFO] ✅ Window done: 0 passed
[INFO] ⏳ stars:>50 language:Kotlin fork:false archived:false created:2008-06-14T00:00:04..2008-08-13T00:00:04 → 0 repos
[INFO] 🔎 Remaining: 21 | Resets in 0.9 min
[INFO] ✅ Window done: 0 passed
[INFO] ⏳ stars:>50 language:Ko